## AggregatorX

This notebook presents the basic usage of the package AggregatorX.

## Loading the package

First we load the necessary packages. The aggregatorX package is loaded with `using AggregatorX`. The software is presently just a module and not registered as a package so some extra code is added here to ensure that the package is not loaded twice which can result in some unexpected bahavior.

In [ ]:
using Pkg

In [ ]:
Pkg.activate(joinpath(@__DIR__)) 

In [ ]:
Pkg.develop(path="..")

In [ ]:
Pkg.add("HiGHS")
Pkg.add("JuMP")
Pkg.add("Plots")

In [ ]:
relpath = "../" # Relative path from this file to where the AggregatorX module is stored.

loaded = false
for n in names(Main)    
    if n == :AggregatorX
        global loaded = true
    end
end
if !loaded # Errors occur if module is loaded multiple times
    include(relpath * "src/AggregatorX.jl")
    using .AggregatorX
end

In [ ]:
using AggregatorX

A choice of solver must be made and loaded. We use the HiGHS package here.

In [ ]:
import HiGHS

The `JuMP` package is loaded by the AggregatorX module so it is not strictly necessary. But to avoid having to call JuMP functions through the AggregatorX module we also include it in the Main scope. We also load `Plots` for visualization

In [ ]:
using JuMP
using Plots

## Building the system
The basic idea of AggregatorX is that the system we wish to investigate can be modelled by individual components that exchange energy through connections. The components can buy or sell energy to different markets (including balancing markets, both capacity and activation markets). Each component defines constraints that model their physical behavior (losses, generation, storage capacities, maximum output power etc.). For more details, refer to the document "Mathematical description".

The system and all its necessary parameters is described in a file in a JSON format, for this first example, in the file `ffr2.json`. The command `buildaggregator()` reads the information in this file and converts this to internal objects used in AggregatorX. This objects are stored in the dictionary `aggregator` returned from the function.

In [ ]:
# Build AggregatorX objects
f = joinpath(@__DIR__, "data", "ffr2.json")
sys, aggregator = buildaggregator(f);

We can inspect the contents of the aggregator object. This object stores variables that are references to variables in the optimization problem and we will use this to investigate the reuslts later.

In [ ]:
aggregator

We see here the classifiction of the main types of components: Resources, Markets, Nodes and Groups. In addition we have Connections which defines how the components are connected and TimeStruct which defines the unit of the time steps (here the TimeStruct is IndexedTimeStruct so each time step is just represented by a dimensionless index).

## Running the optimization

Now we have all the information of the system in the aggregator object and we simply need to run the optimization problem with `optimizeaggregator()`. This function takes the `aggregator` object and a choice of optimizer as input. The function returns and optimizer JuMP model.

In [ ]:
# Build JuMP modell and solve
opt = HiGHS.Optimizer
model = optimizeaggregator(aggregator,opt)

## Exploring the results

The different components in the system (these are coded using Julia structs) has fields that store references to variables in the optimization model, and can be used to query these. Let us for example look at the positions (ie how much is bought and sold) in the different markets. First we get the components which represent the different markets:

In [ ]:
markets = aggregator["Market"]
m = markets[1]
da = markets[2]
ffr = markets[3];

References to the optimization variables are stored in various fields. These fields can store arrays or dictionaries depending on the component. One can get the fieldnames from a component by `fieldnames(type)`, where `type` is the name of the type. E.g. for a FFRProfil component,

In [ ]:
fieldnames(FFRProfil)

We can then query these values using the JuMP `value()` command (note the use of the broadcasting operator (.) to get the alle the values of a vector). For the first two markets the values are stored in dictionaries with keys 1 and 8 (id of the connected component). The last component stores the variable references directly in an array.

In [ ]:
mp = value.(m.power[1])
dap = value.(da.power[8])
ffrc = value.(ffr.up_capacity);

We can then plot and investigate the relationship between the variables.

In [ ]:
scatter(ffrc)
plot!(mp)
plot!(dap)

Now this was just to demonstrate the basic functions so we won't try to interpret this data, but let's move on to some more interesting examples.

## Simple spot price optimization with a battery

Here, we have a simple system consisting of a fixed load which has to be served, a market where you can buy energy and a battery to store it. The battery enebles us to buy energy at times when it is cheap and supply it to the load when the market price is high.

In [ ]:
# Rune the model
f = joinpath(@__DIR__, "data", "fixed_load_spot_battery.json")
sys, aggregator = buildaggregator(f);
model = optimizeaggregator(aggregator,opt)

In [ ]:
aggregator

In [ ]:
# Extract the components
N = aggregator["TimeStruct"].periods
t = [x for x = 1:N]
markets = aggregator["Market"]
resources = aggregator["Resource"]
m = markets[1]
l = resources[1] # load
b = resources[2]; # Battery

In [ ]:
mp = value.(m.power[1])
mprice = m.price
bsoc = value.(b.state_of_charge)
fl = l.load;

In [ ]:
plot(t,[mp mprice bsoc fl], label = ["Market quantity" "Market price" "Battery SoC" "Load demand"])
ylims!(0,10)

### Interpretation

The purple curve is the demand while the orange curve is the price. The blue curve is the quanity of energy bought from the market, we see that this goes to zero when the price becomes very high. To compensate for this we see that the battery (green curve) charges before the expensive period and discharges during the expensive period.

# Variable loads with varying utility

In this example we have three loads which can adjust their loads. Each load has an upper and lower bound (which can vary between time steps) and a certain value/utility is placed on each load. As long as the day-ahead price is lower then the utility, the load will draw the maximum load (upper bound) while if the price is higher than the utility it will draw the minimum load (lower bound).

In [ ]:
# Run the model
f = joinpath(@__DIR__, "data", "multiple_variable_load.json")
sys, aggregator = buildaggregator(f);
model = optimizeaggregator(aggregator,opt)

In [ ]:
# Extract the components
N = aggregator["TimeStruct"].periods
t = [x for x = 1:N]
da = get_component(1, aggregator)
daprice = da.price
vl1 = get_component(3, aggregator)
vl2 = get_component(4, aggregator)
vl3 = get_component(5, aggregator)
vl1power = value.(vl1.power[6]);
vl2power = value.(vl2.power[7]);
vl3power = value.(vl3.power[8]);
vl2ub = value.(vl2.upper_bound);

In [ ]:
linestyle = [:dashdot :solid :solid :solid :dash]
linewidth = [1 2 2 2 1.5]
label = ["Day-ahead price" "VL1 with utility 1.5" "VL2 with utility 2.5" "VL3 with utility 3.5" "VL2 upper bound"]
plot(t, [daprice vl1power vl2power vl3power vl2ub], label = label, linestyle = linestyle, linewidth = linewidth)
ylims!(-1,5)
# VL2 has upperbound = [1,2,3,1,2,3,1,2]
# VL3 has lowerbound = 1

### Interpretation

The blue dashed curve represents the day-ahead price. The three variable loads have varying utility (ie how much we are willing to pay for them). Once the market price exceeds the utility, the variable load is reduced to its minimum.

## FFR market
Let us now look at a simple which includes an FFR market. The system includes a battery and a charger and

In [ ]:
# Run the model
f = joinpath(@__DIR__, "data", "ffr3.json")
sys, aggregator = buildaggregator(f);
model = optimizeaggregator(aggregator,opt)

In [ ]:
solution_summary(model)

In [ ]:
# Extract the components
N = aggregator["TimeStruct"].periods
t = [x for x = 1:N]
da = get_component(3, aggregator)
da = get_component(3, aggregator);
sm = get_component(4, aggregator);
ffr = get_component(5, aggregator);
sb = get_component(2, aggregator);

daprice = da.price
smprice = sm.price
dapower = value.(da.power[8]); # node is 8
smpower = value.(sm.power[1]);
ffrup = value.(ffr.up_capacity);
soc = value.(sb.state_of_charge);

In [ ]:
println("FFR price is " * string(ffr.price[1]))
linestyle = [:solid :solid]
linewidth = [2 2]
label = ["Day-ahead price" "SM price" "DA power" "SM power" "FFR reserved capacity" "State of charge"]
plot(t, [daprice smprice dapower smpower ffrup soc], label = label, linestyle = linestyle, linewidth = linewidth)
ylims!(-0.2, 4)

### Interpretation
We see that the FFR reserved capacity is 2 in the morning. This is provided by the capacity of the charger to stop charging or for the battery to discharge (except for the first hour where the FFR capacity is provided by the charger and the charging of the battery). At 8h the FFR is disarmed and since the DA price is larger than the selling price (2) the charger switches of. At 13h the DA price goes down the charger starts to charge again. At 22h the FFR 

## Tariff

In [ ]:
# Run the model
f = joinpath(@__DIR__, "data", "lineartariff-test1.json")
sys, aggregator = buildaggregator(f);
model = optimizeaggregator(aggregator,opt)

In [ ]:
is_solved_and_feasible(model)

In [ ]:
N = aggregator["TimeStruct"].periods
t = [x for x = 1:N]
da = get_component(1, aggregator);
sm = get_component(5, aggregator);
lt = get_component(2, aggregator);
daprice = da.price
smprice = sm.price
ltprice = lt.price
dapower = value.(da.power[2]);
linewidth = 2
label = ["DA price" "DA power" "Market price" "Tariff"]
plot(t, [daprice dapower smprice ltprice], label=label, linewidth=linewidth)
ylims!(0,5)

# FCR

In [ ]:
# Run the model
f = joinpath(@__DIR__, "data", "fcr-activation.json")
sys, aggregator = buildaggregator(f);
model = optimizeaggregator(aggregator,opt)

In [ ]:
is_solved_and_feasible(model)

In [ ]:
da = get_component(3, aggregator);
sc = get_component(1, aggregator);
fcr = get_component(5, aggregator);
sb = get_component(2, aggregator);
n = get_component(6, aggregator);

DA: 1, 1

SM: 0.5, 0.5

FCR-capacity: 10, 10

FCR-activation: 3, 4

df = 0.05, 0.05


In [ ]:
value.(da.power[6])

In [ ]:
value.(sc.power[4])

In [ ]:
value.(fcr.up_capacity) # =down_capacity

In [ ]:
value.(fcr.down_activation)

In [ ]:
value.(sb.state_of_charge)

In [ ]:
value.(n.power[2])

In [ ]:
da =  -1*1 
market = 0.5*0.5 + 1*0.5
fcrcapacity = 10*1 + 10*1
fcractivation = -3*0.5 + -4*0.5
objective = da + market + fcrcapacity + fcractivation

In [ ]:
objective_value(model)